In [1]:
import json
import os
import tabulate

In [3]:
def get_best_perf_from_json(json_file_path):
    with open(json_file_path, 'r') as file:
        json_data = json.load(file)
        results = {}
        for benchmark_name, benchmark_data in json_data.items():
            best_perf_for_current_benchmark = 0
            for data in benchmark_data:
                perf = data['metrics'][0]['output_throughput']

                if perf > best_perf_for_current_benchmark:
                    best_perf_for_current_benchmark = perf
            results[benchmark_name] = best_perf_for_current_benchmark
    return results
    
    

In [8]:
ROOT = "/Users/franklee/Documents/Projects/Development/My-Plot/examples/data"

# itearte over the folders in the root
for folder in os.listdir(ROOT):
    if os.path.isdir(os.path.join(ROOT, folder)):
        # iterate over teh json files in the folder
        for file in os.listdir(os.path.join(ROOT, folder)):
            if file.endswith(".jsonl"):
                # get the best performance from the json file
                best_perf = get_best_perf_from_json(os.path.join(ROOT, folder, file))
                
                # print it as a table
                data = [
                    (k, v) for k, v in best_perf.items()
                ]
                with open(f"./perfs/{folder}-{file.replace('.jsonl', '.txt')}", "w") as f:
                    headers = [f"{folder}-{file}", "Throughput"]
                    print(tabulate.tabulate(data, headers=headers, tablefmt="grid"), file=f)

                if 'coder' in folder:
                    pass
                else:
                    print(f"===== {folder} {file} =====")
                    print(f"{best_perf['mtbench']:.1f} & 1 & {best_perf['gpqa']:.1f} & 1 & {best_perf['financeqa']:.1f} & 1")
                    print(f"{best_perf['livecodebench']:.1f} & 1 & {best_perf['humaneval']:.1f} & 1 & {best_perf['gsm8k']:.1f} & 1 & {best_perf['math500']:.1f} & 1")


===== nex-30b-a3b no-eagle.jsonl =====
1452.3 & 1 & 1548.9 & 1 & 1425.5 & 1
1521.2 & 1 & 1436.6 & 1 & 1013.8 & 1 & 1531.4 & 1
===== nex-30b-a3b nex-n1_results_20251219_061146.jsonl =====
2215.5 & 1 & 2572.7 & 1 & 1895.8 & 1
2247.5 & 1 & 2504.0 & 1 & 1313.4 & 1 & 2630.7 & 1
===== qwen3-235b-a22b no-eagle.jsonl =====
529.9 & 1 & 563.2 & 1 & 539.5 & 1
598.2 & 1 & 553.1 & 1 & 469.1 & 1 & 587.4 & 1
===== qwen3-235b-a22b qwen3-235b-lmsys_results_20251218_141805.jsonl =====
642.7 & 1 & 716.7 & 1 & 689.4 & 1
803.8 & 1 & 889.9 & 1 & 697.0 & 1 & 821.8 & 1
===== qwen3-235b-a22b qwen3-235b-spec-bundle_results_20251218_203529.jsonl =====
814.5 & 1 & 826.5 & 1 & 889.0 & 1
1155.7 & 1 & 1267.5 & 1 & 758.3 & 1 & 1399.2 & 1
===== qwen3-30b-a3b no-eagle.jsonl =====
1341.3 & 1 & 1410.4 & 1 & 1320.1 & 1
1492.6 & 1 & 1366.6 & 1 & 1071.3 & 1 & 1469.0 & 1
===== qwen3-30b-a3b qwen3-30b-a3b-spec-bundle_results_20251217_083105.jsonl =====
2086.1 & 1 & 2341.3 & 1 & 1779.0 & 1
3413.0 & 1 & 3070.0 & 1 & 1499.6 & 1 

In [3]:
LINES = """
\multirow{3}*{Llama-3.3-70B}    & -           & \multirow{3}*{4} & 560.9 & 1 & 561.0 & 1 & 453.2 & 1 & 567.4 & 1 \\
                                & Existing    &                  & 1303.4 & 1 & 1282.8 & 1 & 521.5 & 1 & 1122.2 & 1 \\
                                & Ours        &                  & 1459.4 & 1 & 1506.0 & 1 & 722.0 & 1 & 1524.9 & 1 \\
"""

results = []
for line in LINES.split("\n"):
    # split the line by \\
    parts = line.strip().split("&")
    
    if len(parts) > 4:
        data = [float(parts[-8]), float(parts[-6]), float(parts[-4]), float(parts[-2])]
        results.append(data)

# compute speedup
speedups = []
for result in results:
    speedup = [
        val / results[0][i]
        for i, val in enumerate(result)
    ]
    speedups.append(speedup)

for val in speedups:
    print(val)


[1.0, 1.0, 1.0, 1.0]
[2.3237653770725624, 2.2866310160427807, 1.1507060900264785, 1.9777934437786395]
[2.601889819932252, 2.6844919786096257, 1.593115622241836, 2.687522030313712]
